# 02 - Download and audit public FVC data

Run cells in order on a Google Colab kernel. CPU is sufficient; this notebook does not train a model.

The official archive contains FVC2000, FVC2002 and FVC2004. Only real sensor images (DB1-DB3) are extracted. DB4 is synthetic and excluded.

Persistent archive and reports: Google Drive / MyDrive / FingerprintResearch. Extracted images: temporary Colab disk. Rerun after a runtime reset to restore them. Downloads resume from the local partial file after connection errors; rerun the download cell without resetting the runtime. Partial downloads do not survive runtime deletion.

**Research limits:** no image-level dry/wet labels are inferred; no image is assumed to be clean ground truth. Finger IDs are scoped to year and database. Subject IDs and cross-dataset identity independence are not established. No train/test split or test-image filtering is performed here.

Official source: http://bias.csr.unibo.it/fvc2004/download.asp . Public download does not imply permission to redistribute biometric images. Check source terms before redistribution. The archive is approximately 830 MiB; allow several GB of runtime disk and about 1 GB of Drive storage.

In [ ]:
import hashlib
import importlib.util
import json
import platform
import re
import shutil
import sys
import urllib.request
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Select Kernel > Colab before running this notebook.')

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

print('Python:', sys.version)
print('Platform:', platform.platform())
print('This stage uses CPU only. No training or GPU allocation is requested by the code.')

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
ARCHIVE_DIR = DRIVE_ROOT / 'archives'
REPORT_DIR = DRIVE_ROOT / 'reports' / 'fvc_audit_v1'
DATA_ROOT = LOCAL_ROOT / 'data'
for directory in (ARCHIVE_DIR, REPORT_DIR, DATA_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = ('https://static-content.springer.com/esm/'
               'chp%3A10.1007%2F978-3-030-83624-5_4/MediaObjects/'
               '74034_3_En_4_MOESM1_ESM.zip')
EXPECTED_BYTES = 869928818
EXPECTED_MD5 = '0412227e3a8c99ab9b0022b8aef9912d'
ARCHIVE_PATH = ARCHIVE_DIR / 'fvc2000_2002_2004_official.zip'
print('Persistent project:', DRIVE_ROOT)
print('Temporary images:', DATA_ROOT)
print('Free runtime disk (GiB):', round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 2))

In [ ]:
import subprocess
import time

def file_digest(path, algorithm):
    digest = hashlib.new(algorithm)
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def resume_download(url, partial, expected_bytes):
    curl = shutil.which('curl')
    if curl is None:
        raise RuntimeError('curl is required for resumable downloads but is not installed.')
    retryable_codes = {5, 6, 7, 18, 28, 35, 52, 55, 56, 92}
    for attempt in range(1, 7):
        size = partial.stat().st_size if partial.exists() else 0
        if size == expected_bytes:
            return
        if size > expected_bytes:
            raise RuntimeError(f'Partial file is oversized: {partial}. Inspect it before replacing it.')
        print(f'Attempt {attempt}/6: resuming at {size / 2**20:.1f} MiB', flush=True)
        result = subprocess.run([
            curl, '--fail', '--location', '--show-error', '--progress-bar',
            '--continue-at', '-', '--connect-timeout', '30',
            '--speed-limit', '1024', '--speed-time', '60', '--max-time', '900',
            '--output', str(partial), url,
        ], check=False)
        size = partial.stat().st_size if partial.exists() else 0
        if result.returncode == 0 and size == expected_bytes:
            return
        if result.returncode == 0:
            raise RuntimeError(f'Server returned an unexpected file size: {size}. Partial file retained.')
        if result.returncode not in retryable_codes:
            raise RuntimeError(
                f'curl failed with exit code {result.returncode}. Partial file retained at {partial}. '
                'See curl error above; HTTP or range errors require investigation.'
            )
        print(f'Connection interrupted (curl {result.returncode}); retained {size / 2**20:.1f} MiB.')
        if attempt < 6:
            time.sleep(min(5 * attempt, 30))
    raise RuntimeError(f'Download interrupted after 6 attempts. Rerun this cell to resume: {partial}')

if not ARCHIVE_PATH.exists():
    partial = LOCAL_ROOT / 'fvc_download.zip.part'
    resume_download(ARCHIVE_URL, partial, EXPECTED_BYTES)
    if file_digest(partial, 'md5') != EXPECTED_MD5:
        raise RuntimeError('Archive checksum mismatch. Do not extract; investigate the source.')
    pending_drive = ARCHIVE_PATH.with_suffix('.zip.part')
    shutil.copyfile(partial, pending_drive)
    if file_digest(pending_drive, 'md5') != EXPECTED_MD5:
        raise RuntimeError('Drive copy checksum mismatch. Final archive was not published.')
    pending_drive.replace(ARCHIVE_PATH)
    partial.unlink()

if ARCHIVE_PATH.stat().st_size != EXPECTED_BYTES:
    raise RuntimeError('Cached archive has unexpected size. Inspect it before replacing it.')
if file_digest(ARCHIVE_PATH, 'md5') != EXPECTED_MD5:
    raise RuntimeError('Cached archive checksum mismatch. Do not use this archive.')
archive_sha256 = file_digest(ARCHIVE_PATH, 'sha256')
print('Archive verified:', ARCHIVE_PATH)
print('SHA256:', archive_sha256)
print('MD5 is used only for source-file integrity, not as a security guarantee.')

In [ ]:
MEMBER_PATTERN = re.compile(
    r'^74034_3_En_4_MOESM1_ESM/(FVC2000|FVC2002|FVC2004)/Dbs/'
    r'DB([123])_([AB])/([0-9]+)_([0-9]+)\.(tif|bmp)$', re.IGNORECASE
)
expected_counts = {
    (year, db, subset): 800 if subset == 'A' else 80
    for year in ('FVC2000', 'FVC2002', 'FVC2004')
    for db in ('DB1', 'DB2', 'DB3')
    for subset in ('A', 'B')
}
records = []
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    selected = []
    counts = Counter()
    for member in archive.infolist():
        match = MEMBER_PATTERN.fullmatch(member.filename)
        if match:
            year, db_num, subset, finger, impression, extension = match.groups()
            year, subset = year.upper(), subset.upper()
            counts[(year, f'DB{db_num}', subset)] += 1
            selected.append((member, year, f'DB{db_num}', subset, int(finger), int(impression)))
    if dict(counts) != expected_counts:
        raise RuntimeError(f'Unexpected archive image counts: {dict(counts)}')
    for member, year, db, subset, finger, impression in selected:
        filename = PurePosixPath(member.filename).name
        relative = Path(year) / f'{db}_{subset}' / filename
        destination = DATA_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open('wb') as output:
            shutil.copyfileobj(source, output)
        records.append({
            'dataset': year, 'db': db, 'official_subset': subset,
            'finger_id': finger, 'impression_id': impression,
            'identity_key': f'{year}_{db}_{finger:03d}',
            'relative_path': relative.as_posix(),
            'condition': 'unknown', 'split': 'unassigned',
        })

manifest = pd.DataFrame(records).sort_values(
    ['dataset', 'db', 'finger_id', 'impression_id']
).reset_index(drop=True)
if manifest.duplicated(['identity_key', 'impression_id']).any():
    raise RuntimeError('Duplicate identity/impression keys detected.')
identity_sizes = manifest.groupby('identity_key').size()
if len(manifest) != 7920 or len(identity_sizes) != 990 or not identity_sizes.eq(8).all():
    raise RuntimeError('Unexpected number of images, identities or impressions.')
print('Extracted 7,920 real images; 990 dataset-scoped finger IDs; 8 impressions per ID.')
print('CRC is checked by zipfile while reading each selected archive member.')

In [ ]:
audited = []
for record in manifest.to_dict('records'):
    path = DATA_ROOT / record['relative_path']
    with Image.open(path) as image:
        image.load()
        width, height = image.size
        mode = image.mode
        extrema = image.convert('L').getextrema()
    if min(width, height) <= 0 or extrema[0] == extrema[1]:
        raise RuntimeError(f'Empty or constant image: {path}')
    audited.append({**record, 'width': width, 'height': height,
                    'mode': mode, 'sha256': file_digest(path, 'sha256')})
manifest = pd.DataFrame(audited)
duplicates = manifest[manifest.duplicated('sha256', keep=False)]
summary = manifest.groupby(['dataset', 'db', 'official_subset']).agg(
    images=('relative_path', 'size'), fingers=('identity_key', 'nunique')
).reset_index()

manifest.to_csv(REPORT_DIR / 'manifest_real_images.csv', index=False)
summary.to_csv(REPORT_DIR / 'counts.csv', index=False)
duplicates.to_csv(REPORT_DIR / 'exact_duplicate_images.csv', index=False)
provenance = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_url': ARCHIVE_URL, 'archive_bytes': ARCHIVE_PATH.stat().st_size,
    'archive_sha256': archive_sha256, 'archive_md5': EXPECTED_MD5,
    'python': sys.version, 'pillow': Image.__version__, 'pandas': pd.__version__,
    'images': len(manifest), 'scoped_finger_ids': len(identity_sizes),
    'exact_duplicate_rows': len(duplicates),
    'dry_wet_labels': 'not available or inferred',
    'split_status': 'unassigned; no training or selection performed',
    'local_data_root': str(DATA_ROOT),
}
(REPORT_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(summary.to_string(index=False))
print('Exact duplicate rows:', len(duplicates))
print('Reports saved:', REPORT_DIR)
if not duplicates.empty:
    print('WARNING: Inspect duplicate report before splitting; no files have been removed.')

In [ ]:
examples = manifest.groupby(['dataset', 'db'], sort=True).head(1)
figure, axes = plt.subplots(3, 3, figsize=(10, 11))
for axis, record in zip(axes.flat, examples.to_dict('records')):
    with Image.open(DATA_ROOT / record['relative_path']) as image:
        axis.imshow(image.convert('L'), cmap='gray', vmin=0, vmax=255)
    axis.set_title(f"{record['dataset']} {record['db']} | {record['width']}x{record['height']}")
    axis.axis('off')
figure.suptitle('First image per database - not a quality-selected sample')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'database_overview.png', dpi=150)
plt.show()
print('Data preparation complete. Next: review audit, then lock splits and matching protocol.')
print('Save this notebook in VS Code. Drive contains archive/reports, not this local notebook automatically.')